# A3 — Watching the Machine
**Your name:**  Pawat (Palm) Punnaket
**Date:**  3 November 2026

This notebook is your A3 deliverable. Fill in each section below. Commit it to your `hcde-410` repo **as you work** — your commit history is part of what you hand in.

> **In this assignment, AI is your subject, not your author.** Do the analysis and writing yourself.

## 1. Your prior practice

*How do you currently use LLMs in your own work, and what do you assume about their reliability? A few sentences.*

I use LLM quite often mainly for quick research or coding. The two models I mainly use is ChatGPT and Claude more specifically Claude Code. I think that reliablity for research depends on the scale. Like an quick easy question I would put it at near perfect reliability but when it comes to deep research or asking about information thats is more niche or strictly guarded LLMs can be quite unreliable. On the other hand, for coding work, I have not had any issues. The output is near perfect everytime. The only issue is that the coding style can be different and not as efficient as how I would normally write code.

## 2. Reproducibility probe

Pick one real task an LLM could help with in data research (write code for a cleaning step, summarize three sources, draft a qualitative codebook, explain a statistical method). Choose one with a **checkable** answer.

Run your prompt **3 times in one model** and **once in a second model**. Log every run in the table below — paste each output verbatim (you may trim a very long one, but keep it faithful).

In [8]:
import pandas as pd

# The exact prompt used for every run (same text each time).
PROMPT = """I have a CSV `bg_hourly_2019.csv` with columns: `date, bgt_north_of_ne_70th_total, ped_south, ped_north, bike_north, bike_south, hour, weekday, bikes, peds`. Each row is one hour of 2019. Write pandas code that finds which hour of the day has the highest average number of bikes on weekdays (Mon–Fri) versus weekends, and print both answers."""

# One row per run.
runs = pd.DataFrame([
    {"run": 1, "prompt": PROMPT, "model": "GPT-6 Astra", "version": "medium",
     "timestamp": "2026-11-03 16:30", 
     "output": """import pandas as pd

df = pd.read_csv("bg_hourly_2019.csv")
df["date"] = pd.to_datetime(df["date"])

# Monday=0, ..., Sunday=6
is_weekday = df["date"].dt.dayofweek < 5

for label, mask in [("Weekdays", is_weekday), ("Weekends", ~is_weekday)]:
    averages = df.loc[mask].groupby("hour")["bikes"].mean()
    peak = averages.max()
    peak_hours = averages[averages == peak].index  # Include any ties
    hours = ", ".join(f"{int(hour):02d}:00" for hour in peak_hours)

    print(f"{label}: {hours} — average {peak:.2f} bikes")""",
     "verified": "correct",
     "notes": "First run everything was fine the output was correct"},  # 

    {"run": 2, "prompt": PROMPT, "model": "GPT-6 Astra", "version": "medium",
     "timestamp": "2026-11-03 16:34",
     "output": """import pandas as pd

df = pd.read_csv("bg_hourly_2019.csv")
df["date"] = pd.to_datetime(df["date"])

is_weekday = df["date"].dt.dayofweek < 5  # Monday=0, Sunday=6

for label, mask in [("Weekdays", is_weekday), ("Weekends", ~is_weekday)]:
    averages = df.loc[mask].groupby("hour")["bikes"].mean()
    peak_average = averages.max()
    peak_hours = averages[averages == peak_average].index

    hours = ", ".join(f"{int(hour):02d}:00" for hour in peak_hours)
    print(f"{label}: {hours} — average {peak_average:.2f} bikes")""",
     "verified": "correct",
     "notes": "Same logic as run 1, but variable names changed (peak → peak_average) and the ties comment was removed"},  

    {"run": 3, "prompt": PROMPT, "model": "GPT-6 Astra", "version": "medium",
     "timestamp": "2026-11-03 16:38",  
     "output": """import pandas as pd

df = pd.read_csv("bg_hourly_2019.csv")
df["date"] = pd.to_datetime(df["date"])

# Derive weekdays from dates: Monday = 0, Sunday = 6.
is_weekday = df["date"].dt.dayofweek < 5

for label, mask in [("Weekdays", is_weekday), ("Weekends", ~is_weekday)]:
    hourly_avg = df.loc[mask].groupby("hour")["bikes"].mean()
    peak_hour = hourly_avg.idxmax()
    print(
        f"{label}: {int(peak_hour):02d}:00 "
        f"({hourly_avg.loc[peak_hour]:.2f} bikes on average)"
    )""",
     "verified": "correct",
     "notes": "This version uses idxmax() to find the peak hour and provides a more detailed output with the average number of bikes"},  # TODO: your notes on this run
    {"run": 4, "prompt": PROMPT, "model": "Claude Opus 5.5", "version": "medium",
     "timestamp": "2026-11-03 16:40",  
     "output": """import pandas as pd

df = pd.read_csv("bg_hourly_2019.csv", parse_dates=["date"])

# Monday=0 ... Sunday=6, so 5 and 6 are the weekend
df["is_weekend"] = df["date"].dt.dayofweek >= 5

# Average bikes for each hour of the day, split into weekdays and weekends
avg = (
    df.groupby(["is_weekend", "hour"])["bikes"]
      .mean()
      .unstack(level=0)
      .rename(columns={False: "weekday", True: "weekend"})
)

for label in ["weekday", "weekend"]:
    peak_hour = avg[label].idxmax()
    print(f"{label.capitalize()} peak hour: {peak_hour}:00 "
          f"(avg {avg[label].max():.1f} bikes)")""",
     "verified": "correct",
     "notes": "This version uses a different approach to find the peak hour and provides a more detailed output"}, 
], columns=["run", "prompt", "model", "version", "timestamp", "output", "verified", "notes"])

runs

,run,prompt,model,version,timestamp,output,verified,notes
0,1,I have a CSV `bg_hourly_2019.csv` with columns...,GPT-6 Astra,medium,2026-11-03 16:30,"import pandas as pd\n\ndf = pd.read_csv(""bg_ho...",correct,First run everything was fine the output was c...
1,2,I have a CSV `bg_hourly_2019.csv` with columns...,GPT-6 Astra,medium,2026-11-03 16:34,"import pandas as pd\n\ndf = pd.read_csv(""bg_ho...",correct,"Same logic as run 1, but variable names change..."
2,3,I have a CSV `bg_hourly_2019.csv` with columns...,GPT-6 Astra,medium,2026-11-03 16:38,"import pandas as pd\n\ndf = pd.read_csv(""bg_ho...",correct,This version uses idxmax() to find the peak ho...
3,4,I have a CSV `bg_hourly_2019.csv` with columns...,Claude Opus 5.5,medium,2026-11-03 16:40,"import pandas as pd\n\ndf = pd.read_csv(""bg_ho...",correct,This version uses a different approach to find...


In [9]:
# A couple of quick reads on your own log (optional, but nice — your log is data too):
if len(runs):
    print("runs logged:", len(runs))
    print("distinct outputs:", runs["output"].nunique())
    print(runs["verified"].value_counts())

runs logged: 4
distinct outputs: 4
verified
correct    4
Name: count, dtype: int64


**What varied, and what held up?** *Summarize the divergence across runs and models, and how you verified each answer against ground truth.*

Overall, the four runs got exactly the same answers but there are some slight differences. The answers were all consistent with Weekdays: 17:00, avg 111.30 bikes and Weekends: 13:00, avg 145.46 bikes (after rounding). The first 3 GPT runs had the same logic given the same prompt. The difference is variable names and run 3 handled the tie handling differently. Runs 1–2 would list all tied hours while run 3 would only listed the one top hour.

For the Claude run, it used a different approach instead of a loop and also rounded differently to only 1 decimals instead of GPT's two decimal. More specifically, Claude added an 'is_weekend' column, and grouped the data differently. It turned the result into a table with 'weekday' column and 'weekend' column (unstack). Then it picked the top hour in each column with 'idxmax()'. The GPT runs looped twice filtering weekdays and weekend.

I confirmed the answer by loading the bg_hourly_2019.csv and checked that all data was present and that is was correct. I then ran each model's code on the same file and compared what it printed with that answer. All 4 matched.

I used the code below to run and check so that I can compare the answers with the LLM.

In [10]:
d = pd.read_csv("../A1/bg_hourly_2019.csv")
weekend = d["weekday"].isin(["Saturday", "Sunday"])
for label, mask in [("Weekdays", ~weekend), ("Weekends", weekend)]:
    avg = d[mask].groupby("hour")["bikes"].mean()
    print(label, avg.idxmax(), round(avg.max(), 2))

Weekdays 17 111.3
Weekends 13 145.46


## 3. Observation

Watch 1–2 people do a comparable task with an LLM. **Ask permission first; don't record audio or video, and don't collect anything personal or sensitive** — this is a classroom exercise, not published research.

Take field notes on specific moments — how they prompt, when they trust vs. doubt, whether and how they verify, and where the model misleads them.

### Field notes

I watched my roommate do research on a topic of his choosing (the topic is not recorded here). He used GPT-6 Astra (medium) for this task. I asked his permission first and did not record audio or video.

- **00:00** Started with instructions as detailed as possible and told GPT to ask questions if more information was needed. He uploaded a PDF about the topic. I beleive the PDF has some background knowledge of his project so that the model will be on the same track. The model read the PDF and replied that it was ready for next steps.
- **01:15** Asked GPT his questions. GPT responded with information on the topic but provided no sources.
- **01:40** Asked GPT "Are you sure? pls double check." GPT thought for roughly 10 seconds and replied that it was sure, restating the information. He later checked the information online and it was the same as the information on a Wikipedia page meaning.
- **02:10** Asked GPT for the source of the information. GPT repeated the answer, phrased slightly differently, but the numbers still matched its original answer. GPT also now attached a link to the Wikipedia page and another source with the same data. He clicked the links, both links were active and usable. 
- **06:42** After writing about his findings in a document, he came back and asked a follow-up question. GPT answered in roughly 10 seconds without citing sources or saying where the data came from.
- **07:19** He asked GPT to "Always attach the source where you got the information." GPT responded with the same answer and attached a link to the source. The link was active and usable.


In [11]:
# Group your field notes into a few themes.
codes = pd.DataFrame([
    {"time": "00:00", "observation": "Gave detailed instructions and told GPT to ask questions if it needed more info", "theme": "setup"},
    {"time": "01:15", "observation": "Asked his questions; GPT answered with no sources", "theme": "trusting"},
    {"time": "01:40", "observation": "Asked 'Are you sure? pls double check'; GPT said it was sure and restated the answer", "theme": "doubting"},
    {"time": "02:10", "observation": "Asked for the source; GPT repeated the answer and attached two working links", "theme": "verification"},
    {"time": "06:42", "observation": "Asked a follow-up; GPT answered without citing sources", "theme": "trusting"},
    {"time": "07:19", "observation": "Told GPT to 'Always attach the source'; GPT attached a working link", "theme": "prompt repair"},
], columns=["time", "observation", "theme"])

codes


,time,observation,theme
0,00:00,Gave detailed instructions and told GPT to ask...,setup
1,01:15,Asked his questions; GPT answered with no sources,trusting
2,01:40,Asked 'Are you sure? pls double check'; GPT sa...,doubting
3,02:10,Asked for the source; GPT repeated the answer ...,verification
4,06:42,Asked a follow-up; GPT answered without citing...,trusting
5,07:19,Told GPT to 'Always attach the source'; GPT at...,prompt repair


## 4. Synthesis & reflection

*Bring the probe and the observation together into one account. What does this mean for trustworthy, reproducible research — and what will you change about how you use and verify AI in your own work?*

_Write here._

## Competency declaration & AI use

- **Competencies demonstrated** — in a sentence each, say which of the three this work shows, and where the evidence is: *qualitative & mixed-methods inquiry*, *reproducible analysis*, *ethical & societal reasoning*.
- **AI use** — note any AI use in producing this notebook, per the course policy. Remember: AI is your subject here, not your author.